In [52]:
import pandas as pd
import numpy as np
import optuna
import warnings
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
from optuna.integration import LightGBMPruningCallback
import xgboost as xgb
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [53]:
#importazione dei dataset
raw_data = fetch_california_housing()
data = pd.DataFrame(raw_data.data, columns=raw_data.feature_names)
data['MedHouseVal'] = raw_data.target

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


In [ ]:
#rimozione outlier
numeric_cols = data.select_dtypes(include=["float", "int"]).columns.tolist()

for col in numeric_cols:
    Q1 = data[col].quantile(0.3)
    Q3 = data[col].quantile(0.7)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

    data[col] = np.where(data[col].between(lower, upper), data[col], np.nan)

data = data.dropna()

#features combinate
data['Ave_Room_Bed'] = data['AveRooms'] / data['AveBedrms']
data['HouseRooms'] = data['HouseAge'] / data['AveRooms']
data['Rooms_per_Person'] = data['AveRooms'] / data['AveOccup']

In [55]:
'''
def objective(trial):
    # Split interno per validazione durante il tuning
    X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)

    # Spazio di ricerca
    param = {
        'verbosity': 0,
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse', # Metrica per early stopping e pruning
        'tree_method': 'hist',
        
        # --- Struttura ---
        # California ha interazioni complesse, permettiamo alberi più profondi
        'max_depth': trial.suggest_int('max_depth', 2, 20),
        # Importante per evitare overfitting su outlier di prezzo
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 25),
        'gamma': trial.suggest_float('gamma', 1e-7, 1.5, log=True),
        
        # --- Randomness ---
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        
        # --- Regolarizzazione ---
        # Fondamentale nella regressione per non inseguire i prezzi estremi
        'lambda': trial.suggest_float('lambda', 1e-9, 15.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-9, 15.0, log=True),
        
        # --- Learning ---
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.6),
    }

    # Pruning basato su RMSE
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "validation-rmse")

    model = xgb.train(
        param,
        dtrain,
        num_boost_round=1000,
        evals=[(dvalid, "validation")],
        early_stopping_rounds=100,
        callbacks=[pruning_callback],
        verbose_eval=False
    )

    # Optuna deve minimizzare RMSE
    preds = model.predict(dvalid)
    rmse = np.sqrt(mean_squared_error(y_valid, preds))
    return rmse
'''


'\ndef objective(trial):\n    # Split interno per validazione durante il tuning\n    X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)\n\n    dtrain = xgb.DMatrix(X_train, label=y_train)\n    dvalid = xgb.DMatrix(X_valid, label=y_valid)\n\n    # Spazio di ricerca\n    param = {\n        \'verbosity\': 0,\n        \'objective\': \'reg:squarederror\',\n        \'eval_metric\': \'rmse\', # Metrica per early stopping e pruning\n        \'tree_method\': \'hist\',\n\n        # --- Struttura ---\n        # California ha interazioni complesse, permettiamo alberi più profondi\n        \'max_depth\': trial.suggest_int(\'max_depth\', 2, 20),\n        # Importante per evitare overfitting su outlier di prezzo\n        \'min_child_weight\': trial.suggest_int(\'min_child_weight\', 1, 25),\n        \'gamma\': trial.suggest_float(\'gamma\', 1e-7, 1.5, log=True),\n\n        # --- Randomness ---\n        \'subsample\': trial.suggest_float(\'

In [58]:

def objective(trial):
    # Split interno per validazione durante il tuning
    X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)

    # Spazio di ricerca
    param = {
        'verbosity': 0,
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse', # Metrica per early stopping e pruning
        'tree_method': 'hist',
        
        # --- Struttura ---
        # California ha interazioni complesse, permettiamo alberi più profondi
        'max_depth': trial.suggest_int('max_depth', 2, 20),
        # Importante per evitare overfitting su outlier di prezzo
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 25),
        'gamma': trial.suggest_float('gamma', 1e-7, 1.5, log=True),
        
        # --- Randomness ---
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        
        # --- Regolarizzazione ---
        # Fondamentale nella regressione per non inseguire i prezzi estremi
        'lambda': trial.suggest_float('lambda', 1e-9, 15.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-9, 15.0, log=True),
        
        # --- Learning ---
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.6),
    }

    model = xgb.cv(
        params=param,
        dtrain=dtrain,
        num_boost_round=3000,
        nfold=5,
        early_stopping_rounds=100,
        verbose_eval=False
    )

    best_rmse = model["test-rmse-mean"].iloc[-1]
    return best_rmse


In [59]:
#X, y = data.data, data.target
X = data.drop(columns=["MedHouseVal"])
y= data["MedHouseVal"]

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset Shape: {X.shape}")
print(f"Target medio: {np.mean(y):.2f} (centinaia di k$)")

# ==========================================
# 2. Baseline (Senza Tuning)
# ==========================================
print("\n--- Training Modello Baseline (Default) ---")
dtrain_full = xgb.DMatrix(X_train_full, label=y_train_full)
dtest = xgb.DMatrix(X_test, label=y_test)

# Parametri standard di default
params_base = {
    'objective': 'reg:squarederror', # Regressione
    'tree_method': 'hist',
    'random_state': 42
}

model_base = xgb.train(params_base, dtrain_full, num_boost_round=100)
preds_base = model_base.predict(dtest)

# Calcolo RMSE Baseline
rmse_base = np.sqrt(mean_squared_error(y_test, preds_base))
print(f"Baseline RMSE: {rmse_base:.4f}")


# ==========================================
# 3. Ottimizzazione con Optuna
# ==========================================
print("\n--- Inizio Tuning con Optuna ---")

# Creazione Studio (Minimizzare RMSE)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=200, timeout=600)

print(f"\nMiglior RMSE trovato da Optuna (Validation): {study.best_value:.4f}")
print("Migliori parametri:", study.best_params)


# ==========================================
# 4. Training Modello Finale (Best Params + Shrinkage)
# ==========================================
print("\n--- Training Modello Finale Ottimizzato ---")

best_params = study.best_params
best_params['objective'] = 'reg:squarederror'
best_params['eval_metric'] = 'rmse'
best_params['tree_method'] = 'hist'

# STRATEGIA "SHRINKAGE" (Raffiniamo la discesa del gradiente)
# Abbassiamo il learning rate trovato per maggiore precisione finale
best_params['learning_rate'] = best_params['learning_rate'] * 0.5 
# Aumentiamo gli alberi di conseguenza (safety buffer alto, early stopping gestirà il resto)
num_round_final = 5000 

model_opt = xgb.train(
    best_params, 
    dtrain_full, 
    num_boost_round=num_round_final,
    evals=[(dtest, "test")], # Test set usato SOLO per fermare il training al punto giusto
    early_stopping_rounds=100,
    verbose_eval=False
)

# ==========================================
# 5. Analisi Risultati
# ==========================================
print("\n--- CONFRONTO FINALE ---")

preds_opt = model_opt.predict(dtest)
rmse_opt = np.sqrt(mean_squared_error(y_test, preds_opt))

# R2 Score (coefficiente di determinazione)
r2_base = r2_score(y_test, preds_base)
r2_opt = r2_score(y_test, preds_opt)

print(f"RMSE Baseline:    {rmse_base:.4f}")
print(f"RMSE Ottimizzato: {rmse_opt:.4f}")
print(f"--> Miglioramento Errore: {rmse_base - rmse_opt:.4f} (Minore è meglio)")
print("-" * 30)
print(f"R2 Score Baseline:    {r2_base:.4f}")
print(f"R2 Score Ottimizzato: {r2_opt:.4f}")
print(f"--> Varianza Spiegata Extra: +{(r2_opt - r2_base)*100:.2f}%")

Dataset Shape: (13561, 11)
Target medio: 1.85 (centinaia di k$)

--- Training Modello Baseline (Default) ---


[I 2025-12-09 15:56:28,500] A new study created in memory with name: no-name-6f34e34b-ae7a-4aee-b639-282e0a8b17e2


Baseline RMSE: 0.3548

--- Inizio Tuning con Optuna ---


[I 2025-12-09 15:56:35,195] Trial 0 finished with value: 0.3685824035343137 and parameters: {'max_depth': 13, 'min_child_weight': 10, 'gamma': 0.01950696736781367, 'subsample': 0.9706896729029745, 'colsample_bytree': 0.998446196490038, 'lambda': 2.8341889030842336e-06, 'alpha': 0.0170337708504397, 'learning_rate': 0.045639977797611024}. Best is trial 0 with value: 0.3685824035343137.
[I 2025-12-09 15:56:38,542] Trial 1 finished with value: 0.3753122067820108 and parameters: {'max_depth': 6, 'min_child_weight': 4, 'gamma': 0.0023989326665202096, 'subsample': 0.6802724031163863, 'colsample_bytree': 0.9475318290771388, 'lambda': 0.0005738861937986768, 'alpha': 3.2971928979516787, 'learning_rate': 0.30949677472896836}. Best is trial 0 with value: 0.3685824035343137.
[I 2025-12-09 15:56:48,725] Trial 2 finished with value: 0.37445734041905576 and parameters: {'max_depth': 15, 'min_child_weight': 5, 'gamma': 2.80689080953014e-07, 'subsample': 0.8704953196331897, 'colsample_bytree': 0.9180103


Miglior RMSE trovato da Optuna (Validation): 0.3525
Migliori parametri: {'max_depth': 7, 'min_child_weight': 7, 'gamma': 0.00020274676329353587, 'subsample': 0.9498287415841411, 'colsample_bytree': 0.8770878213992869, 'lambda': 1.58393444158607e-05, 'alpha': 4.636874131770243e-08, 'learning_rate': 0.038093048415023936}

--- Training Modello Finale Ottimizzato ---

--- CONFRONTO FINALE ---
RMSE Baseline:    0.3548
RMSE Ottimizzato: 0.3292
--> Miglioramento Errore: 0.0256 (Minore è meglio)
------------------------------
R2 Score Baseline:    0.8194
R2 Score Ottimizzato: 0.8445
--> Varianza Spiegata Extra: +2.51%


In [ ]:
'''
def objective(trial):
    # Split interno per validazione durante il tuning
    X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)

    # Spazio di ricerca
    param = {
        'verbosity': 0,
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse', # Metrica per early stopping e pruning
        'tree_method': 'hist',
        
        # --- Struttura ---
        # California ha interazioni complesse, permettiamo alberi più profondi
        'max_depth': trial.suggest_int('max_depth', 2, 20),
        # Importante per evitare overfitting su outlier di prezzo
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 25),
        'gamma': trial.suggest_float('gamma', 1e-7, 1.5, log=True),
        
        # --- Randomness ---
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        
        # --- Regolarizzazione ---
        # Fondamentale nella regressione per non inseguire i prezzi estremi
        'lambda': trial.suggest_float('lambda', 1e-9, 15.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-9, 15.0, log=True),
        
        # --- Learning ---
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.6),
    }

    # Pruning basato su RMSE
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "validation-rmse")

    model = xgb.train(
        param,
        dtrain_pseudo,
        num_boost_round=1000,
        evals=[(dvalid, "validation")],
        early_stopping_rounds=100,
        callbacks=[pruning_callback],
        verbose_eval=False
    )

    # Optuna deve minimizzare RMSE
    preds = model.predict(dvalid)
    rmse = np.sqrt(mean_squared_error(y_valid, preds))
    return rmse
'''

In [61]:

def objective(trial):
    # Split interno per validazione durante il tuning
    X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)

    # Spazio di ricerca
    param = {
        'verbosity': 0,
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse', # Metrica per early stopping e pruning
        'tree_method': 'hist',
        
        # --- Struttura ---
        # California ha interazioni complesse, permettiamo alberi più profondi
        'max_depth': trial.suggest_int('max_depth', 2, 20),
        # Importante per evitare overfitting su outlier di prezzo
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 25),
        'gamma': trial.suggest_float('gamma', 1e-7, 1.5, log=True),
        
        # --- Randomness ---
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        
        # --- Regolarizzazione ---
        # Fondamentale nella regressione per non inseguire i prezzi estremi
        'lambda': trial.suggest_float('lambda', 1e-9, 15.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-9, 15.0, log=True),
        
        # --- Learning ---
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.6),
    }

    model = xgb.cv(
        params=param,
        dtrain= dtrain_pseudo,
        num_boost_round=3000,
        nfold=5,
        early_stopping_rounds=100,
        verbose_eval=False
    )

    best_rmse = model["test-rmse-mean"].iloc[-1]
    return best_rmse


In [62]:
#X, y = data.data, data.target
X = data.drop(columns=["MedHouseVal"])
y= data["MedHouseVal"]

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset Shape: {X.shape}")
print(f"Target medio: {np.mean(y):.2f} (centinaia di k$)")

# ==========================================
# 2. Baseline (Senza Tuning)
# ==========================================
print("\n--- Training Modello Baseline (Default) ---")
dtrain_full = xgb.DMatrix(X_train_full, label=y_train_full)
dtest = xgb.DMatrix(X_test, label=y_test)

# Parametri standard di default
params_base = {
    'objective': 'reg:squarederror', # Regressione
    'tree_method': 'hist',
    'random_state': 42
}

model_base = xgb.train(params_base, dtrain_full, num_boost_round=100)
preds_base = model_base.predict(dtest)

# Calcolo RMSE Baseline
rmse_base = np.sqrt(mean_squared_error(y_test, preds_base))
print(f"Baseline RMSE: {rmse_base:.4f}")

# ==========================================
# 2.5. Pseudo-Labeling
# ==========================================

X_unlabeled = X.sample(200, replace=False).copy()
# fingi di non conoscere MedHouseVal
# X_unlabeled = df_unlabeled[features]  # <-- per uso reale

# Scala i dati non etichettati
X_unl_scaled = scaler.transform(X_unlabeled)

# Predici pseudo-label usando il modello baseline
d_unl = xgb.DMatrix(X_unlabeled)
pseudo_labels = model_base.predict(d_unl)

print(f"Creati {len(pseudo_labels)} pseudo-labels")

# Costruisci dataset esteso
X_pseudo = pd.concat([X_train_full, X_unlabeled], axis=0)
y_pseudo = pd.concat([y_train_full, pd.Series(pseudo_labels, index=X_unlabeled.index)])

print("Dataset esteso pronto:")
print(f" - Originale: {len(X_train_full)}")
print(f" - Pseudo-labels: {len(X_unlabeled)}")
print(f" - Totale: {len(X_pseudo)}")

# DMatrix esteso
dtrain_pseudo = xgb.DMatrix(X_pseudo, label=y_pseudo)


# ==========================================
# 3. Ottimizzazione con Optuna
# ==========================================
print("\n--- Inizio Tuning con Optuna ---")

# Creazione Studio (Minimizzare RMSE)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=200, timeout=600)

print(f"\nMiglior RMSE trovato da Optuna (Validation): {study.best_value:.4f}")
print("Migliori parametri:", study.best_params)


# ==========================================
# 4. Training Modello Finale (Best Params + Shrinkage)
# ==========================================
print("\n--- Training Modello Finale Ottimizzato ---")

best_params = study.best_params
best_params['objective'] = 'reg:squarederror'
best_params['eval_metric'] = 'rmse'
best_params['tree_method'] = 'hist'

# STRATEGIA "SHRINKAGE" (Raffiniamo la discesa del gradiente)
# Abbassiamo il learning rate trovato per maggiore precisione finale
best_params['learning_rate'] = best_params['learning_rate'] * 0.5 
# Aumentiamo gli alberi di conseguenza (safety buffer alto, early stopping gestirà il resto)
num_round_final = 5000

model_opt = xgb.train(
    best_params, 
    dtrain_pseudo,
    num_boost_round=num_round_final,
    evals=[(dtest, "test")], # Test set usato SOLO per fermare il training al punto giusto
    early_stopping_rounds=100,
    verbose_eval=False
)

# ==========================================
# 5. Analisi Risultati
# ==========================================
print("\n--- CONFRONTO FINALE ---")

preds_opt = model_opt.predict(dtest)
rmse_opt = np.sqrt(mean_squared_error(y_test, preds_opt))

# R2 Score (coefficiente di determinazione)
r2_base = r2_score(y_test, preds_base)
r2_opt = r2_score(y_test, preds_opt)

print(f"RMSE Baseline:    {rmse_base:.4f}")
print(f"RMSE Ottimizzato: {rmse_opt:.4f}")
print(f"--> Miglioramento Errore: {rmse_base - rmse_opt:.4f} (Minore è meglio)")
print("-" * 30)
print(f"R2 Score Baseline:    {r2_base:.4f}")
print(f"R2 Score Ottimizzato: {r2_opt:.4f}")
print(f"--> Varianza Spiegata Extra: +{(r2_opt - r2_base)*100:.2f}%")

Dataset Shape: (13561, 11)
Target medio: 1.85 (centinaia di k$)

--- Training Modello Baseline (Default) ---


[I 2025-12-09 16:14:01,725] A new study created in memory with name: no-name-15d6c1f8-1072-46fb-b001-caabf63f7767


Baseline RMSE: 0.3548
Creati 200 pseudo-labels
Dataset esteso pronto:
 - Originale: 10848
 - Pseudo-labels: 200
 - Totale: 11048

--- Inizio Tuning con Optuna ---


[I 2025-12-09 16:14:03,157] Trial 0 finished with value: 0.3662220937004581 and parameters: {'max_depth': 10, 'min_child_weight': 18, 'gamma': 0.08361369107292208, 'subsample': 0.9981614266501795, 'colsample_bytree': 0.6907945107695478, 'lambda': 0.013111514449291772, 'alpha': 0.018456292388011043, 'learning_rate': 0.2727354942472456}. Best is trial 0 with value: 0.3662220937004581.
[I 2025-12-09 16:14:05,454] Trial 1 finished with value: 0.39076144046512423 and parameters: {'max_depth': 9, 'min_child_weight': 3, 'gamma': 0.4150690461962725, 'subsample': 0.8826170046928549, 'colsample_bytree': 0.8994634696637469, 'lambda': 0.00127978020189115, 'alpha': 1.2775812037513898e-09, 'learning_rate': 0.29777989661573284}. Best is trial 0 with value: 0.3662220937004581.
[I 2025-12-09 16:14:18,121] Trial 2 finished with value: 0.36311601880344163 and parameters: {'max_depth': 2, 'min_child_weight': 25, 'gamma': 7.668412041748635e-07, 'subsample': 0.8731836654229158, 'colsample_bytree': 0.8072318


Miglior RMSE trovato da Optuna (Validation): 0.3403
Migliori parametri: {'max_depth': 16, 'min_child_weight': 23, 'gamma': 0.00012941183979607294, 'subsample': 0.8832430152395088, 'colsample_bytree': 0.7627727728568615, 'lambda': 1.0894634562674758e-06, 'alpha': 9.689071159971518e-08, 'learning_rate': 0.01735810925617593}

--- Training Modello Finale Ottimizzato ---

--- CONFRONTO FINALE ---
RMSE Baseline:    0.3548
RMSE Ottimizzato: 0.3323
--> Miglioramento Errore: 0.0225 (Minore è meglio)
------------------------------
R2 Score Baseline:    0.8194
R2 Score Ottimizzato: 0.8416
--> Varianza Spiegata Extra: +2.22%


In [ ]:
def objective(trial):

    params = {
        # --- Config Hardware ---
        'device_type': 'cpu', 
        'n_jobs': -1,
        'verbose': -1,

        # --- Config velocità ---
        'max_bin': 255,          
        'bagging_freq': 1,
        'bagging_fraction': 0.8,

        # --- Parametri Modello ---
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        
        # --- Search Space (Ridotto per convergere prima) ---
        'n_estimators': 1000, # Abbassato il tetto massimo
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.3, log=True), # LR più alto = impara prima
        'num_leaves': trial.suggest_int('num_leaves', 20, 100), # Meno foglie = alberi più semplici
        'max_depth': trial.suggest_int('max_depth', 3, 8),      # Alberi meno profondi
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 100, 1000),
        
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }

    model = lgb.LGBMRegressor(**params)
    # Pruning basato su RMSE
    pruning_callback = LightGBMPruningCallback(trial, "rmse")

    model.fit(
        X_train_full, y_train_full,
        eval_set=[(X_test, y_test)],
        eval_metric='rmse',
        callbacks=[
            # Smetti subito se non migliori per 10 round (molto aggressivo)
            lgb.early_stopping(stopping_rounds=10, verbose=False),
            pruning_callback
        ]
    )

    # Optuna deve minimizzare RMSE
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    return rmse


In [ ]:
#X, y = data.data, data.target
X = data.drop(columns=["MedHouseVal"])
y= data["MedHouseVal"]

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset Shape: {X.shape}")
print(f"Target medio: {np.mean(y):.2f} (centinaia di k$)")

# ==========================================
# 2. Baseline (Senza Tuning)
# ==========================================
dtrain_full = lgb.Dataset(X_train_full, label=y_train_full)
dtest = lgb.Dataset(X_test, label=y_test)

params_base = {
    'objective': 'regression',
    'metric': 'rmse',
    'seed': 42
}

model_base = lgb.train(params_base, dtrain_full, num_boost_round=100)

preds_base = model_base.predict(X_test)

# Calcolo RMSE Baseline
rmse_base = np.sqrt(mean_squared_error(y_test, preds_base))
print(f"Baseline RMSE: {rmse_base:.4f}")


# ==========================================
# 3. Ottimizzazione con Optuna
# ==========================================
print("\n--- Inizio Tuning con Optuna ---")

# Creazione Studio (Minimizzare RMSE)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=200, timeout=600)

print(f"\nMiglior RMSE trovato da Optuna (Validation): {study.best_value:.4f}")
print("Migliori parametri:", study.best_params)


# ==========================================
# 4. Training Modello Finale (Best Params + Shrinkage)
# ==========================================
print("\n--- Training Modello Finale Ottimizzato ---")

best_params = study.best_params
best_params['objective'] = 'regression'
best_params['eval_metric'] = 'rmse'
best_params['tree_method'] = 'hist'

# STRATEGIA "SHRINKAGE" (Raffiniamo la discesa del gradiente)
# Abbassiamo il learning rate trovato per maggiore precisione finale
best_params['learning_rate'] = best_params['learning_rate'] * 0.5 
# Aumentiamo gli alberi di conseguenza (safety buffer alto, early stopping gestirà il resto)
num_round_final = 5000 

model_opt = lgb.train(
    best_params,
    dtrain_full,
    num_boost_round=num_round_final,
    valid_sets=[dtest],
    valid_names=["test"],
    callbacks=[lgb.early_stopping(stopping_rounds=100)]
)

# ==========================================
# 5. Analisi Risultati
# ==========================================
print("\n--- CONFRONTO FINALE ---")

preds_opt = model_opt.predict(X_test)
rmse_opt = np.sqrt(mean_squared_error(y_test, preds_opt))

# R2 Score (coefficiente di determinazione)
r2_base = r2_score(y_test, preds_base)
r2_opt = r2_score(y_test, preds_opt)

print(f"RMSE Baseline:    {rmse_base:.4f}")
print(f"RMSE Ottimizzato: {rmse_opt:.4f}")
print(f"--> Miglioramento Errore: {rmse_base - rmse_opt:.4f} (Minore è meglio)")
print("-" * 30)
print(f"R2 Score Baseline:    {r2_base:.4f}")
print(f"R2 Score Ottimizzato: {r2_opt:.4f}")
print(f"--> Varianza Spiegata Extra: +{(r2_opt - r2_base)*100:.2f}%")


Dataset Shape: (13561, 11)
Target medio: 1.85 (centinaia di k$)


[I 2025-12-09 15:44:31,216] A new study created in memory with name: no-name-a5ba09f7-24a8-4cf4-b3a4-492052925ee0


Baseline RMSE: 0.3491

--- Inizio Tuning con Optuna ---


[I 2025-12-09 15:44:31,998] Trial 0 finished with value: 0.35901448050362844 and parameters: {'learning_rate': 0.14362039563520684, 'num_leaves': 40, 'max_depth': 8, 'min_data_in_leaf': 320, 'reg_alpha': 0.01398283840596186, 'reg_lambda': 0.022502417306488852}. Best is trial 0 with value: 0.35901448050362844.
[I 2025-12-09 15:44:32,497] Trial 1 finished with value: 0.38526415372443745 and parameters: {'learning_rate': 0.16329805032845746, 'num_leaves': 32, 'max_depth': 3, 'min_data_in_leaf': 640, 'reg_alpha': 3.8955147199046145, 'reg_lambda': 8.956495780445014}. Best is trial 0 with value: 0.35901448050362844.
[I 2025-12-09 15:44:33,184] Trial 2 finished with value: 0.3585621993851808 and parameters: {'learning_rate': 0.16667734564107636, 'num_leaves': 75, 'max_depth': 8, 'min_data_in_leaf': 259, 'reg_alpha': 0.007334617241025389, 'reg_lambda': 0.016993907731378124}. Best is trial 2 with value: 0.3585621993851808.
[I 2025-12-09 15:44:33,995] Trial 3 finished with value: 0.3618357945192


Miglior RMSE trovato da Optuna (Validation): 0.3516
Migliori parametri: {'learning_rate': 0.2244129939089908, 'num_leaves': 88, 'max_depth': 6, 'min_data_in_leaf': 186, 'reg_alpha': 0.07796343926851752, 'reg_lambda': 0.005263834885935298}

--- Training Modello Finale Ottimizzato ---
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[810]	test's l2: 0.117188

--- CONFRONTO FINALE ---
RMSE Baseline:    0.3491
RMSE Ottimizzato: 0.3423
--> Miglioramento Errore: 0.0068 (Minore è meglio)
------------------------------
R2 Score Baseline:    0.8252
R2 Score Ottimizzato: 0.8319
--> Varianza Spiegata Extra: +0.67%


**RISULTATO**

RMSE Baseline:    0.4718
RMSE Ottimizzato: 0.4300

param = {
        'verbosity': 0,
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse', # Metrica per early stopping e pruning
        'tree_method': 'hist',
        
        # --- Struttura ---
        # California ha interazioni complesse, permettiamo alberi più profondi
        'max_depth': trial.suggest_int('max_depth', 2, 15),
        # Importante per evitare overfitting su outlier di prezzo
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        
        # --- Randomness ---
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        
        # --- Regolarizzazione ---
        # Fondamentale nella regressione per non inseguire i prezzi estremi
        'lambda': trial.suggest_float('lambda', 1e-9, 25.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-9, 25.0, log=True),
        
        # --- Learning ---
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
    }

study.optimize(objective, n_trials=70, timeout=600)

=============================================================================== <br>
RMSE Baseline:    0.4718
RMSE Ottimizzato: 0.4310

con scaler

param = {
        'verbosity': 0,
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse', # Metrica per early stopping e pruning
        'tree_method': 'hist',
        
        # --- Struttura ---
        # California ha interazioni complesse, permettiamo alberi più profondi
        'max_depth': trial.suggest_int('max_depth', 2, 20),
        # Importante per evitare overfitting su outlier di prezzo
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 25),
        'gamma': trial.suggest_float('gamma', 1e-9, 1.5, log=True),
        
        # --- Randomness ---
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        
        # --- Regolarizzazione ---
        # Fondamentale nella regressione per non inseguire i prezzi estremi
        'lambda': trial.suggest_float('lambda', 1e-6, 20.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-6, 20.0, log=True),
        
        # --- Learning ---
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.8),
    }

study.optimize(objective, n_trials=200, timeout=600)

=============================================================================== <br>

RMSE Baseline:    0.3522
RMSE Ottimizzato: 0.3379

con scaler

study.optimize(objective, n_trials=200, timeout=600)

param = {
        'verbosity': 0,
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse', # Metrica per early stopping e pruning
        'tree_method': 'hist',
        
        # --- Struttura ---
        # California ha interazioni complesse, permettiamo alberi più profondi
        'max_depth': trial.suggest_int('max_depth', 2, 20),
        # Importante per evitare overfitting su outlier di prezzo
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 25),
        'gamma': trial.suggest_float('gamma', 1e-7, 1.5, log=True),
        
        # --- Randomness ---
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        
        # --- Regolarizzazione ---
        # Fondamentale nella regressione per non inseguire i prezzi estremi
        'lambda': trial.suggest_float('lambda', 1e-9, 15.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-9, 15.0, log=True),
        
        # --- Learning ---
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.6),
    }

#rimozione outlier
numeric_cols = data.select_dtypes(include=["float", "int"]).columns.tolist()

for col in numeric_cols:
    Q1 = data[col].quantile(0.3)
    Q3 = data[col].quantile(0.7)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

    data[col] = np.where(data[col].between(lower, upper), data[col], np.nan)

data = data.dropna()

#features combinate
data['Ave_Room_Bed'] = data['AveRooms'] / data['AveBedrms']
data['HouseRooms'] = data['HouseAge'] / data['AveRooms']

Miglior RMSE trovato da Optuna (Validation): 0.3544
Migliori parametri: {'max_depth': 13, 'min_child_weight': 23, 'gamma': 3.8838420870962745e-06, 'subsample': 0.7570628531701007, 'colsample_bytree': 0.9109736816364161, 'lambda': 0.02307771384295654, 'alpha': 0.9454327224813569, 'learning_rate': 0.06885420667941514}

=============================================================================== <br>
RMSE Baseline:    0.3548
RMSE Ottimizzato: 0.3297

con scaler

study.optimize(objective, n_trials=200, timeout=600)

param = {
        'verbosity': 0,
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse', # Metrica per early stopping e pruning
        'tree_method': 'hist',
        
        # --- Struttura ---
        # California ha interazioni complesse, permettiamo alberi più profondi
        'max_depth': trial.suggest_int('max_depth', 2, 20),
        # Importante per evitare overfitting su outlier di prezzo
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 25),
        'gamma': trial.suggest_float('gamma', 1e-7, 1.5, log=True),
        
        # --- Randomness ---
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        
        # --- Regolarizzazione ---
        # Fondamentale nella regressione per non inseguire i prezzi estremi
        'lambda': trial.suggest_float('lambda', 1e-9, 15.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-9, 15.0, log=True),
        
        # --- Learning ---
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.6),
    }

#rimozione outlier
numeric_cols = data.select_dtypes(include=["float", "int"]).columns.tolist()

for col in numeric_cols:
    Q1 = data[col].quantile(0.3)
    Q3 = data[col].quantile(0.7)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

    data[col] = np.where(data[col].between(lower, upper), data[col], np.nan)

data = data.dropna()

#features combinate
data['Ave_Room_Bed'] = data['AveRooms'] / data['AveBedrms']
data['HouseRooms'] = data['HouseAge'] / data['AveRooms']
data['Rooms_per_Person'] = data['AveRooms'] / data['AveOccup']

Miglior RMSE trovato da Optuna (Validation): 0.3490
Migliori parametri: {'max_depth': 6, 'min_child_weight': 10, 'gamma': 5.35007610966066e-05, 'subsample': 0.8096500866788845, 'colsample_bytree': 0.7174314936237725, 'lambda': 2.180030783832174e-06, 'alpha': 9.101912061894957e-09, 'learning_rate': 0.042067598298740405}